# VALORA AI — 02 Modelling
**Project:** Apartment asking-price estimation in Montenegro  
**Input:** `data/processed/valora_apartments_clean.csv` created by `01_EDA.ipynb`.  
I use real online apartment listings, not synthetic examples. All results below are calculated when the notebook runs.

## 1. TL;DR
I compare a simple median-price baseline, Linear Regression, Random Forest, Random Forest with Feature Engineering, and a tuned Random Forest. I evaluate all models on the same held-out test set. **The best model and its scores will be determined after running the code.**

## 2. Recap of the goal
I want to estimate the **asking price in EUR** of an apartment in Montenegro from its area, bedrooms, and city. I compare MAE, RMSE, and R². Lower MAE and RMSE are better. The predictions are not confirmed sale prices or certified valuations.

## 3. Setup & data loading
I import the required libraries and load the cleaned CSV produced by the EDA notebook. The notebook works from the project root in VS Code or Colab. If the file is missing, I need to run EDA first or place its exported CSV in `data/processed/`.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from google.colab import files

# Upload the cleaned dataset from the EDA notebook
uploaded = files.upload()

path = Path(next(iter(uploaded)))
if not path.exists():
    raise FileNotFoundError(f'Missing {path}. Run 01_EDA.ipynb and save its cleaned CSV here.')
df = pd.read_csv(path, low_memory=False)
print('Dataset shape:', df.shape)
display(df.head())
print('Available columns:', df.columns.tolist())

Saving valora_apartments_clean.csv to valora_apartments_clean.csv
Dataset shape: (2435, 61)


,listing_id,izvor,property_type,price_eur,area_m2,cijena_po_m2,sobe,kupatila,lokacija,city,...,Floor,Condition,Year_Built,Price_per_m2_EUR,Data_Source,is_synthetic,price_per_m2_calculated,has_price_and_area,property_type_clean,bedrooms
0,2760812,Realitica,stan,154000.0,59.0,2610.17,1.0,NaN,"Đenovići,",Herceg Novi,...,NaN,NaN,NaN,NaN,NaN,False,2610.169492,True,stan,1.0
1,2812045,Realitica,stan,1520000.0,230.0,6608.70,5.0,NaN,"Meljine,",Herceg Novi,...,NaN,NaN,NaN,NaN,NaN,False,6608.695652,True,stan,5.0
2,2816025,Realitica,stan,320000.0,58.0,5517.24,1.0,NaN,"Kotor, Kotor,",Kotor,...,NaN,NaN,NaN,NaN,NaN,False,5517.241379,True,stan,1.0
3,2832141,Realitica,stan,220000.0,93.0,2365.59,2.0,NaN,"Baošići,",Herceg Novi,...,NaN,NaN,NaN,NaN,NaN,False,2365.591398,True,stan,2.0
4,2832146,Realitica,stan,550000.0,178.0,3089.89,3.0,NaN,"Herceg Novi,",Herceg Novi,...,NaN,NaN,NaN,NaN,NaN,False,3089.887640,True,stan,3.0


Available columns: ['listing_id', 'izvor', 'property_type', 'price_eur', 'area_m2', 'cijena_po_m2', 'sobe', 'kupatila', 'lokacija', 'city', 'naslov', 'url', 'opis', 'provjera', 'data_origin', 'simulated_condition', 'simulated_year_built', 'simulated_floor', 'simulated_land_area_m2', 'data_source', 'document', 'land_area_m2', 'valuation_year', 'price_per_m2', 'review_status', 'city_suggested', 'quality_check', 'possible_duplicate', 'confirmed_city', 'confirmed_area_m2', 'confirmed_price_eur', 'notes', 'verification_status', 'units_in_appraisal', 'source_note', 'original_area_m2', 'original_market_value_eur', 'residential_area_m2', 'auxiliary_area_m2', 'weighted_valuation_area_m2', 'valuation_includes_land', 'price_per_m2_basis', 'source_pages', 'training_status', 'building_area_m2', 'verified_area_m2', 'verified_market_value_eur', 'Listing_ID', 'Property_Type', 'Land_Area_m2', 'Bedrooms', 'Floor', 'Condition', 'Year_Built', 'Price_per_m2_EUR', 'Data_Source', 'is_synthetic', 'price_per_m

## 4. Feature preparation
I check the columns and prepare features that are available before a price is estimated. I use apartment area, bedrooms and cleaned city. I also test `area_squared`, `area_per_bedroom` and `size_group`. I do **not** use `price_per_m2` because it is calculated from the price we are predicting. Missing-value imputation and one-hot encoding are fitted only on the training data inside a pipeline.

In [ ]:
required = ['price_eur', 'area_m2', 'bedrooms', 'city']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Expected columns missing from EDA output: {missing}')
work = df.copy()
for c in ['price_eur', 'area_m2', 'bedrooms']:
    work[c] = pd.to_numeric(work[c], errors='coerce')
work['city'] = work['city'].fillna('Unknown').astype(str).str.strip()
work.loc[work['city'].eq(''), 'city'] = 'Unknown'
# A final check keeps apartment descriptions out of the city feature.
invalid_city = work['city'].str.contains(r'\bstan\s+\d+\s+sob', case=False, na=False)
work.loc[invalid_city, 'city'] = 'Unknown'
work = work.dropna(subset=['price_eur', 'area_m2'])
work = work[(work['price_eur'] > 0) & (work['area_m2'] > 0)].copy()
if 'is_synthetic' in work.columns:
    work = work[~work['is_synthetic'].astype(str).str.lower().isin(['true','1'])].copy()
if 'data_source' in work.columns:
    work = work[work['data_source'].eq('Online listing')].copy()
if 'property_type' in work.columns:
    work = work[work['property_type'].astype(str).str.strip().str.lower().isin(['stan','apartment'])].copy()
print('Rows ready for modelling:', len(work))
print('Unknown city:', (work['city'] == 'Unknown').sum())
display(work[required].isna().sum().to_frame('Missing values'))

def add_features(data):
    data = data.copy()
    data['area_squared'] = data['area_m2'] ** 2
    data['area_per_bedroom'] = data['area_m2'] / data['bedrooms'].replace(0, np.nan)
    data['size_group'] = pd.cut(data['area_m2'], [0, 50, 100, np.inf],
                                labels=['Small', 'Medium', 'Large']).astype(object)
    data['size_group'] = data['size_group'].fillna('Unknown')
    return data

base_features = ['area_m2', 'bedrooms', 'city']
X = work[base_features].copy()
y = work['price_eur'].copy()

## 5. Train/test split
I reserve 20% of the apartments for final testing (`random_state=42`). I do not use the test data for model selection or hyperparameter tuning. All models are evaluated on the same test apartments.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train_fe = add_features(X_train)
X_test_fe = add_features(X_test)
print('Training:', X_train.shape, 'Testing:', X_test.shape)

def make_preprocessor(engineered=False):
    numeric = ['area_m2', 'bedrooms']
    categories = ['city']
    if engineered:
        numeric += ['area_squared', 'area_per_bedroom']
        categories += ['size_group']
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric),
        ('cat', Pipeline([
            ('fill', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore'))
        ]), categories)
    ])

def evaluate(name, model, features):
    predictions = model.predict(features)
    return {'Model': name,
            'MAE (EUR)': mean_absolute_error(y_test, predictions),
            'RMSE (EUR)': np.sqrt(mean_squared_error(y_test, predictions)),
            'R2': r2_score(y_test, predictions)}

## 6. Baseline model
I first predict the median training price for every apartment. This simple baseline helps me check whether machine learning gives a meaningful improvement.

In [ ]:
baseline = DummyRegressor(strategy='median')
baseline.fit(X_train, y_train)
results = [evaluate('Median baseline', baseline, X_test)]
display(pd.DataFrame(results).round(2))

## 7. Candidate models
I train Linear Regression and Random Forest, as in my original Valora notebook. Then I test Random Forest with engineered features. Finally, I use cross-validation **on the training set only** to tune the Random Forest. This avoids selecting settings based on the final test results.

In [ ]:
linear_model = Pipeline([
    ('preprocessing', make_preprocessor()), ('model', LinearRegression())
])
linear_model.fit(X_train, y_train)

forest_model = Pipeline([
    ('preprocessing', make_preprocessor()),
    ('model', RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                                    random_state=42, n_jobs=-1))
])
forest_model.fit(X_train, y_train)

forest_fe = Pipeline([
    ('preprocessing', make_preprocessor(engineered=True)),
    ('model', RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                                    random_state=42, n_jobs=-1))
])
forest_fe.fit(X_train_fe, y_train)

# Tune with cross-validation on training data; test data is not used here.
tuning_pipeline = Pipeline([
    ('preprocessing', make_preprocessor(engineered=True)),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])
param_grid = {
    'model__n_estimators': [100, 150, 200],
    'model__min_samples_leaf': [2, 3, 4],
    'model__max_depth': [None, 15, 20]
}
search = GridSearchCV(tuning_pipeline, param_grid, scoring='neg_mean_absolute_error',
                      cv=KFold(n_splits=3, shuffle=True, random_state=42),
                      n_jobs=1, refit=True)
search.fit(X_train_fe, y_train)
tuned_model = search.best_estimator_
print('Best training CV settings:', search.best_params_)
print('Best cross-validation MAE (EUR):', round(-search.best_score_, 2))

## 8. Evaluation & comparison
I compare all models on the same test set. I use MAE as the main measure because it is easy to understand in euros. The chart shows the average absolute error for each model. The best result is calculated, not assumed.

In [ ]:
models = {
    'Linear Regression': (linear_model, X_test),
    'Random Forest': (forest_model, X_test),
    'Random Forest + Feature Engineering': (forest_fe, X_test_fe),
    'Tuned Random Forest + FE': (tuned_model, X_test_fe)
}
for name, (model, features) in models.items():
    results.append(evaluate(name, model, features))
results_df = pd.DataFrame(results).sort_values('MAE (EUR)').reset_index(drop=True)
display(results_df.round(2))
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='MAE (EUR)', y='Model')
plt.title('Model comparison — test MAE (lower is better)')
plt.xlabel('Mean absolute error (EUR)')
plt.ylabel('Model')
plt.tight_layout()
plt.show()
best_name = results_df.loc[0, 'Model']
print('Lowest test MAE:', best_name)
# Model choice for deployment should ideally be confirmed with further validation.

## 9. Error analysis
I compare actual asking prices with predictions and inspect the largest errors. I also check whether errors vary across cities or apartment sizes. The test set is only used for reporting, not tuning.

In [ ]:
chosen_name = 'Tuned Random Forest + FE'
chosen_model, chosen_X = models[chosen_name]
predictions = chosen_model.predict(chosen_X)
errors = pd.DataFrame({
    'city': X_test['city'].to_numpy(),
    'area_m2': X_test['area_m2'].to_numpy(),
    'bedrooms': X_test['bedrooms'].to_numpy(),
    'actual_price_eur': y_test.to_numpy(),
    'predicted_price_eur': predictions
}, index=X_test.index)
errors['error_eur'] = errors['predicted_price_eur'] - errors['actual_price_eur']
errors['absolute_error_eur'] = errors['error_eur'].abs()
display(errors.sort_values('absolute_error_eur', ascending=False).head(10).round(0))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(errors['actual_price_eur'], errors['predicted_price_eur'], alpha=.45)
lo = min(errors['actual_price_eur'].min(), errors['predicted_price_eur'].min())
hi = max(errors['actual_price_eur'].max(), errors['predicted_price_eur'].max())
axes[0].plot([lo, hi], [lo, hi], '--', color='black')
axes[0].set(title='Actual vs predicted asking prices', xlabel='Actual EUR', ylabel='Predicted EUR')
axes[1].hist(errors['error_eur'], bins=30)
axes[1].set(title='Prediction errors', xlabel='Predicted minus actual (EUR)', ylabel='Number of apartments')
plt.tight_layout(); plt.show()
print('Test errors by city (at least 5 apartments):')
by_city = errors.groupby('city').agg(apartments=('absolute_error_eur','size'),
                                     MAE_EUR=('absolute_error_eur','mean'))
display(by_city[by_city['apartments'] >= 5].sort_values('MAE_EUR', ascending=False).round(0))

## 10. Interpretation
I examine feature importance for the tuned Random Forest. This helps explain which inputs the model uses most. Feature importance is not proof of cause and effect. I also show five test predictions and a separate example of a new apartment.

In [ ]:
feature_names = chosen_model.named_steps['preprocessing'].get_feature_names_out()
importance = chosen_model.named_steps['model'].feature_importances_
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
importance_df = importance_df.sort_values('Importance', ascending=False)
display(importance_df.head(15).round(3))
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x='Importance', y='Feature')
plt.title('Random Forest — feature importance')
plt.tight_layout(); plt.show()
print('Five example predictions from the test set:')
display(errors[['city','area_m2','bedrooms','actual_price_eur','predicted_price_eur']].head(5).round(0))
new_apartment = pd.DataFrame({'area_m2': [65], 'bedrooms': [2], 'city': ['Podgorica']})
estimated_price = chosen_model.predict(add_features(new_apartment))[0]
print(f'Example: 65 m², 2 bedrooms, Podgorica → estimated asking price: €{estimated_price:,.0f}')

## 11. Limitations & ethics
- Listings contain **asking prices**, not verified sale prices.
- City alone does not capture neighborhood, condition, floor, view, building quality or distance to the sea.
- Some city values remain `Unknown`; location data can be incomplete.
- The dataset may contain outdated listings, duplicate properties or biased coverage of expensive areas.
- Synthetic records are excluded from this real-listings model.
- Feature importance is descriptive, not causal.
- These predictions are exploratory and should not replace a professional appraisal.
- A single random test split is not enough to establish reliability across time and locations.

## 12. Conclusion & next steps
After running this notebook, I will report which model has the lowest MAE and whether Feature Engineering and tuning helped. I will not claim an improvement unless the results show it. Next, I want to collect more verified real-property data, improve neighborhood information, review the largest errors, and test the model on newer listings. Later, I would like to build a simple Valora AI price-estimation app.